### Hybrid Search Langchain

In [4]:
import os

from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [5]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [6]:
index_name = "hybrid-search-langchain-pinecone"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )

In [7]:
index = pc.Index(index_name)

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5715.90it/s]


In [9]:
bm25_encoder = BM25Encoder().default()

In [10]:
sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

In [11]:
bm25_encoder.fit(sentences)

100%|██████████| 3/3 [00:00<00:00, 30.00it/s]


In [14]:
bm25_encoder.dump("bm25_values.json")

In [15]:
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [16]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
)

In [17]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:01<00:00,  1.85s/it]


In [18]:
results = retriever.invoke("What city did I visit first?")

In [19]:
results

[Document(metadata={'score': 0.257887483}, page_content='In 2021, I visited New Orleans'),
 Document(metadata={'score': 0.244532719}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.220131129}, page_content='In 2023, I visited Paris')]